# Práctica 1 — Síntesis de voz

**Equipo:** _(poned aquí vuestro número y los nombres)_

Este cuaderno es el punto de partida. Ejecutad las celdas en orden.
Donde pone `# TAREA` tenéis que escribir o decidir algo vosotros;
el resto es andamiaje y funciona tal cual.

Las funciones de apoyo están en `scripts/`. Podéis abrirlas y leerlas:
están comentadas y explican por qué cada medida se toma como se toma.

## 0. Puesta en marcha

Esta celda crea las carpetas, fija `HF_HOME` (la caché por defecto del
workspace es de solo lectura) y deja `scripts/` accesible.
Ejecutadla **siempre la primera**, también al reabrir el cuaderno.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'scripts'))

from tts_util import preparar_entorno, sin_ruido, cargar_frases, resumen_corpus
from tts_util import BASE, AUDIO, RES, MOS, EQUIPO

preparar_entorno()

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import Audio, display

import adaptadores, analisis, evaluacion

---
# Sesión 1 — Corpus y candidatos

## Tarea 1.1 — Corpus de frases

Todas las medidas de la práctica se hacen sobre las mismas 20 frases.
Partid de `frases_ejemplo.tsv`, copiadlo como `frases.tsv` y completadlo
con frases de **vuestro dominio**.

Composición obligatoria: 6 `decl`, 4 `int_abs`, 4 `int_parc`,
3 `cifras`, 3 `siglas`.

La columna `leida` es cómo **debería** pronunciarse la frase, en letras y
sin puntuación. Es la referencia para medir la inteligibilidad en la S3.

In [ ]:
# TAREA 1.1 — ejecutad esto una vez para crear vuestro frases.tsv,
# después editadlo desde el panel de ficheros de JupyterLab.
import shutil
destino = BASE / 'frases.tsv'
if not destino.exists():
    shutil.copy('frases_ejemplo.tsv', destino)
    print(f'Creado {destino}. Editadlo y volved aquí.')
else:
    print(f'{destino} ya existe.')

In [ ]:
frases = cargar_frases()
resumen_corpus(frases)

## Tarea 1.2 — Candidatos

Al menos **tres sistemas** que funcionen, de **al menos dos familias**
distintas. Piper viene preparado; el segundo es un ejemplo que puede o no
funcionar; el tercero lo buscáis vosotros.

Si un candidato no arranca: anotad el error exacto, dedicadle como mucho
30 minutos y descartadlo. Un sistema descartado con su diagnóstico cuenta
como candidato explorado en el informe.

In [ ]:
CANDIDATOS = {}

# Candidato A — Piper (CPU). Descarga la voz la primera vez (~74 MB).
adaptadores.registrar(CANDIDATOS, 'piper_sharvard',
                      adaptadores.crear_piper, 'es_ES-sharvard-medium')

# Candidato B — MMS-TTS (GPU). Si falla, anotad el error y seguid.
adaptadores.registrar(CANDIDATOS, 'mms_spa',
                      adaptadores.crear_mms)

# TAREA 1.2 — Candidato C: buscadlo vosotros y registradlo aquí.
# Si es otra voz de Piper, recordad que cuenta como la MISMA familia.

print()
print('Candidatos activos:', list(CANDIDATOS))

In [ ]:
# Prueba rápida: una frase con cada candidato
for nombre, tts in CANDIDATOS.items():
    ruta = AUDIO / 'prueba' / f'{nombre}.wav'
    tts(frases[0]['texto'], ruta)
    print(nombre)
    display(Audio(str(ruta)))

## Tarea 1.3 — Ficha de cada candidato

Rellenad esta tabla en el informe (aquí podéis ir tomando notas).
Las tres primeras columnas se completan hoy; la calidad, en la S3.

| Criterio | Dónde se mira |
|---|---|
| Licencia y uso comercial | *model card*, fichero LICENSE del repositorio |
| Español: variedad, nº de voces | *model card* y vuestra escucha |
| Memoria ocupada | `torch.cuda.memory_allocated()` o `nvidia-smi` |
| Velocidad (RTF y carga) | tarea 1.4 |
| Mantenimiento | última versión, actividad, dependencias |
| Calidad | sesión 3 |

## Tarea 1.4 — Velocidad

`medir_rtf` separa la carga del modelo del tiempo de síntesis, descarta
una primera llamada de calentamiento y toma la mediana de varias
repeticiones. Las tres cosas hay que saber justificarlas en la defensa.

In [ ]:
filas = []
for nombre, tts in CANDIDATOS.items():
    print(f'Midiendo {nombre} ...')
    for f in analisis.medir_rtf(tts, frases, AUDIO / nombre):
        filas.append(dict(sistema=nombre, **f))

rtf = pd.DataFrame(filas)
rtf.to_csv(RES / 'rtf.csv', index=False)
rtf.groupby('sistema')[['rtf', 'dur']].describe().round(3)

In [ ]:
# Tiempo de CARGA de cada modelo, que se reporta aparte del RTF
_, t_piper = analisis.medir_carga(adaptadores.crear_piper, 'es_ES-sharvard-medium')
print(f'Carga de Piper: {t_piper} s')

# TAREA 1.4 — Repetid la medida de Piper con usar_gpu=True y comparad.
# ¿Gana siempre la GPU con un modelo pequeño? Justificadlo.

---
# Sesión 2 — Control prosódico

La pregunta de esta sesión no es si suena distinto, sino **qué cambia y
qué no** al tocar los parámetros o el texto.

`analisis.analiza(ruta)` devuelve la duración, la F0 mediana, el recorrido
de F0 en semitonos y `final_st`: el movimiento de los últimos 300 ms.
Positivo = termina subiendo; negativo = termina bajando.

## Tarea 2.1 — ¿Es lineal el control de velocidad?

In [ ]:
escalas = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6]
decl = [f for f in frases if f['categoria'] == 'decl']

tts = CANDIDATOS['piper_sharvard']   # solo sistemas con admite_parametros
filas = []
for fr in decl:
    for e in escalas:
        ruta = AUDIO / 'velocidad' / f"{fr['id']}_{e}.wav"
        tts(fr['texto'], ruta, length_scale=e)
        filas.append(dict(id=fr['id'], escala=e, dur=analisis.analiza(ruta)['dur']))

vel = pd.DataFrame(filas)
base = vel[vel.escala == 1.0].set_index('id').dur
vel['dur_rel'] = vel.dur / vel.id.map(base)
vel.groupby('escala').dur_rel.mean().round(3)

In [ ]:
# TAREA 2.1 — Dibujad dur_rel frente a escala junto con la diagonal y = x.
# ¿Se ajusta? ¿Se alargan igual los silencios que los sonidos?
# ¿A partir de qué valor la voz deja de ser aceptable? (escuchadlo)

## Tarea 2.2 — ¿Es determinista el sistema?

In [ ]:
# TAREA 2.2 — Sintetizad la MISMA frase 5 veces en tres condiciones:
#   a) parámetros por defecto
#   b) noise_scale=0.0, noise_w_scale=0.0
#   c) noise_scale=1.0, noise_w_scale=1.0
# Para cada grupo, desviación típica de la duración y del rango_st.
# ¿Con variabilidad cero salen audios idénticos? ¿Suena más monótona?

## Tarea 2.3 — Modalidad: ¿pregunta cuando debe?

Ningún sistema tiene un parámetro «pregunta»: la entonación se infiere de
la puntuación. Comprobad si lo hace bien.

In [ ]:
fr = decl[0]['texto'].rstrip('.')
versiones = {'.': fr + '.', '?': '¿' + fr + '?', '!': '¡' + fr + '!'}

etiquetadas = []
for nombre, tts in CANDIDATOS.items():
    for marca, texto in versiones.items():
        ruta = AUDIO / 'modalidad' / f'{nombre}_{marca.strip(chr(46))or "pt"}.wav'
        tts(texto, ruta)
        a = analisis.analiza(ruta)
        print(f'{nombre:20s} {marca}  final_st = {a["final_st"]:+.2f}')
        if nombre == 'piper_sharvard':
            etiquetadas.append((marca, ruta))

analisis.dibuja_contornos(etiquetadas, 'Modalidad — Piper',
                          RES / 'modalidad_piper.png');

In [ ]:
# TAREA 2.3 — Aplicad lo mismo a vuestras 4 int_abs y 4 int_parc.
# En español la interrogativa absoluta suele terminar en ascenso y la
# parcial no necesariamente: ¿lo reproduce cada sistema?

## Tarea 2.4 — Foco: ¿se puede enfatizar una palabra?

Los sistemas no tienen control de foco. Solo podéis intentarlo desde el
texto: mayúsculas, comas, reordenar, repetir, partir en dos oraciones.
Que no funcione nada es un resultado válido y hay que documentarlo.

In [ ]:
# TAREA 2.4 — Probad al menos tres trucos sobre una frase, poniendo el
# énfasis en tres palabras distintas. Medid F0 máxima y duración de la
# palabra objetivo en cada versión.

## Tarea 2.5 — Normalización de texto

In [ ]:
# Escuchad las frases de las categorías cifras y siglas en los 3 sistemas
for fr in [f for f in frases if f['categoria'] in ('cifras', 'siglas')]:
    print(fr['texto'])
    for nombre in CANDIDATOS:
        display(Audio(str(AUDIO / nombre / f"{fr['id']}.wav")))

In [ ]:
# TAREA 2.5 — Escribid normaliza(texto) para convertir fechas, horas,
# importes y siglas a letras ANTES de sintetizar, y comparad el resultado.

def normaliza(texto):
    # ... vuestro código
    return texto

---
# Sesión 3 — Evaluación de calidad

Tres vías que responden a preguntas distintas. Lo interesante aparece
cuando no coinciden.

## Tarea 3.1 — Inteligibilidad

Se transcribe la voz sintética con el reconocedor de referencia de la
asignatura (el mismo para todos los equipos) y se compara con la columna
`leida`. Ocupa VRAM: cargadlo cuando vayáis a usarlo.

In [ ]:
intel = evaluacion.mide_inteligibilidad(CANDIDATOS, frases)
intel.to_csv(RES / 'inteligibilidad.csv', index=False)
intel.pivot_table(index='categoria', columns='sistema', values='wer').round(3)

**Separad el habla normal de las cifras y siglas.** En esas dos categorías
buena parte del WER no es falta de inteligibilidad, sino desacuerdo entre
vuestra columna `leida` y cómo escribe el reconocedor.

In [ ]:
evaluacion.resumen_wer(intel)

Ahora revisad **a mano** las frases difíciles. Para cada una, decidid si el
error es artefacto de convención (la voz se entendió, se escribió distinto)
o fallo real de síntesis (la voz dijo algo que no era). Esa clasificación
va al informe.

In [ ]:
evaluacion.revisar(intel)

In [ ]:
# TAREA 3.1 — Revisad A MANO las transcripciones de la categoría cifras.
# ¿Cuántos "errores" son en realidad '30' frente a 'treinta'?
intel[intel.categoria == 'cifras'][['sistema', 'referencia', 'hipotesis', 'wer']]

## Tarea 3.2 — MOS

Panel cruzado y ciego: vuestras muestras las puntúan otros equipos.
Elegid 5 frases (al menos una de cada categoría).

In [ ]:
# TAREA 3.2 — poned aquí vuestros 5 ids
elegidas = ['f01', 'f03', 'f04', 'f05', 'f06']

clave = evaluacion.preparar_mos(CANDIDATOS, frases, elegidas)

In [ ]:
# Cuando recibáis las notas (CSV con columnas evaluador, muestra, nota):
# evaluacion.analiza_mos(RES / 'mos_recibidas.csv')

## Tarea 3.3 — Consistencia de la voz

In [ ]:
emb, intra, entre = evaluacion.consistencia(CANDIDATOS, frases)

In [ ]:
# TAREA 3.3 — Comparad la consistencia de las frases interrogativas frente
# a las declarativas. ¿Cambia la "identidad" de la voz cuando pregunta?

## Tarea 3.4 — Reflexión ética

500-800 palabras en el informe, con al menos dos fuentes citadas.
Apoyaos en lo que habéis oído esta semana: ¿podrían estas voces engañar a
alguien? ¿En qué se nota que son sintéticas?

---
## Decisión final

¿Qué sistema usaríais en el asistente de vuestro dominio y por qué?
Ponderad calidad, velocidad, recursos y licencia. «El de mayor MOS» no
basta si es diez veces más lento o su licencia no lo permite.

In [ ]:
# Resumen de todas las medidas, para la discusión del informe
resumen = rtf.groupby('sistema').rtf.median().rename('rtf_mediano').to_frame()
resumen['wer_intel'] = intel.groupby('sistema').wer.mean()
resumen['consistencia'] = pd.Series(intra)
resumen.round(3)